# Listener Prior (Dual Dataset) - Keyterms Prediction (Public Repo, Colab GPU)

This notebook:
- clones the repo into `/content/listener-prior`
- installs dependencies (pins `datasets<4.0.0` so MultiWOZ/DailyDialog script datasets load)
- trains the bi-encoder optimized for **keyterm/keyword prediction** (not full utterance retrieval)
- evaluates on keyterm precision/recall/F1 metrics instead of MRR/Recall
- builds **keyterms/keywords** from retrieved candidates (for Deepgram STT injection)
- writes outputs to Google Drive so they persist

**Key difference**: This notebook optimizes for predicting keyterms/keywords that will appear in the other person's next utterance, rather than predicting the full utterance text.

In [ ]:
# --- CONFIG: set your GitHub repo here ---
REPO_URL = "https://github.com/<YOUR_GITHUB_USERNAME>/<YOUR_REPO_NAME>.git"  # <- edit
PROJECT_DIR = "/content/listener-prior"

In [ ]:
!rm -rf "$PROJECT_DIR"
!git clone "$REPO_URL" "$PROJECT_DIR"
%cd "$PROJECT_DIR"
!ls -la

In [ ]:
# Install deps. We intentionally do NOT pin torch here; Colab already has a CUDA build.
import os
assert 'PROJECT_DIR' in globals() or os.environ.get('PROJECT_DIR'), 'Run the repo setup cell first.'
PROJECT_DIR = globals().get('PROJECT_DIR', os.environ.get('PROJECT_DIR', '/content/listener-prior'))
os.chdir(PROJECT_DIR)
print('CWD:', os.getcwd())
!python -m pip install -U pip
# Force a compatible datasets version: newer datasets drops loading scripts + trust_remote_code.
!grep -v '^torch' requirements.txt > /tmp/requirements_no_torch.txt
!python -m pip install -r /tmp/requirements_no_torch.txt --upgrade
# Optional (fast retrieval indexing)
!python -m pip install faiss-cpu || true
import datasets
print('datasets version:', datasets.__version__)
assert tuple(int(x) for x in datasets.__version__.split(".")[:1]) < (4,), "datasets must be < 4.0.0; restart runtime after install"

In [ ]:
# If the assert above failed, go to Runtime -> Restart runtime, then rerun from the top.
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Hugging Face token (recommended).
# In Colab: Tools -> Secrets -> add HF_TOKEN.
import os
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    token = None

# Clear any stale/expired tokens that might already exist in the environment.
for k in ['HF_TOKEN', 'HUGGINGFACE_HUB_TOKEN', 'HUGGING_FACE_HUB_TOKEN']:
    os.environ.pop(k, None)

if token:
    os.environ['HF_TOKEN'] = token
    os.environ['HUGGINGFACE_HUB_TOKEN'] = token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = token
    print('HF token set from Colab Secrets.')
else:
    print('No HF_TOKEN secret found. Public datasets/models should still work without it.')

## Model Selection

Choose a sentence transformer model for training:

- **Recommended (smaller, faster)**: `sentence-transformers/paraphrase-MiniLM-L3-v2` (~60MB, optimized for semantic similarity)
- **Default**: `sentence-transformers/all-MiniLM-L6-v2` (~80MB, good balance)
- **Larger (better quality)**: `sentence-transformers/all-MiniLM-L12-v2` (~130MB, slower)

The `paraphrase-MiniLM-L3-v2` model is recommended as it's smaller, faster, and performs similarly well for predicting what the other person will say next.

In [ ]:
# Model selector - choose your model
# Recommended: paraphrase-MiniLM-L3-v2 (smaller, faster, similar quality)
MODEL_NAME = "sentence-transformers/paraphrase-MiniLM-L3-v2"  # Recommended: smaller, faster
# MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"  # Default: good balance
# MODEL_NAME = "sentence-transformers/all-MiniLM-L12-v2"  # Larger: better quality, slower

print(f"Selected model: {MODEL_NAME}")
print("\nModel comparison:")
print("  paraphrase-MiniLM-L3-v2: ~60MB, fast, optimized for semantic similarity")
print("  all-MiniLM-L6-v2:         ~80MB, balanced (default)")
print("  all-MiniLM-L12-v2:        ~130MB, better quality, slower")

## Keyterm Configuration

**Keyterm F1@5 and F1@10 Explanation:**
- The model retrieves the top-k predicted keyterm candidates for each conversation history
- **Precision@k**: Of the top k predicted keyterms, how many actually appear in the ground truth?
- **Recall@k**: Of all ground truth keyterms, how many were found in the top k predictions?
- **F1@k**: Harmonic mean of precision and recall (balanced metric)
- **F1@5**: Evaluates top 5 predictions (more focused, higher precision)
- **F1@10**: Evaluates top 10 predictions (broader coverage, higher recall)
- The model with the highest Keyterm F1@10 is selected as the best model

Set the maximum number of keyterms/keywords to extract and use for training:

In [ ]:
# Keyterm extraction parameters
MAX_KEYWORDS = 30  # Maximum number of single-word keywords to extract
MAX_KEYTERMS = 30  # Maximum number of multi-word keyterms (phrases) to extract

print(f"Keyterm Configuration:")
print(f"  Max Keywords: {MAX_KEYWORDS} (single words)")
print(f"  Max Keyterms: {MAX_KEYTERMS} (multi-word phrases)")
print(f"\nNote: These are used for:")
print(f"  - Training: extracting keyterms from target utterances")
print(f"  - Evaluation: comparing predicted vs ground truth keyterms")
print(f"  - Inference: generating keyterms for Deepgram STT injection")

In [ ]:
import datetime
run_id = datetime.datetime.now().strftime('dual_keyterms_%Y%m%d_%H%M%S')
OUTPUT_DIR = f"/content/drive/MyDrive/listener_prior_runs/{run_id}"
print('OUTPUT_DIR:', OUTPUT_DIR)

In [ ]:
# Train optimized for keyterm prediction.
# This script evaluates on keyterm precision/recall/F1 metrics instead of MRR/Recall.
# target_role=SYSTEM means: predict keyterms that will be in the other person's next utterance.
# The model with the highest Keyterm F1@10 on the test set is saved as the best model.
!python scripts/train_dual_epoch_test_keyterms.py \
  --output_dir "$OUTPUT_DIR" \
  --device auto \
  --model_name "$MODEL_NAME" \
  --epochs 6 \
  --batch_size 32 \
  --learning_rate 4.3e-5 \
  --weight_decay 0.01 \
  --adam_beta1 0.95 \
  --adam_beta2 0.98 \
  --adam_eps 1e-8 \
  --grad_accum_steps 2 \
  --warmup_ratio 0.0 \
  --history_turns 6 \
  --target_role SYSTEM \
  --max_keywords "$MAX_KEYWORDS" \
  --max_keyterms "$MAX_KEYTERMS" \
  --val_ratio 0.05 \
  --test_ratio 0.15 \
  --max_dialogs_multiwoz 0 \
  --max_dialogs_dailydialog 0

## View Best Model Performance

Display the test set performance metrics for the best model (selected based on Keyterm F1@10 score).

In [ ]:
# Display best model performance on test set
import json
import os

best_eval_path = os.path.join(OUTPUT_DIR, "best_eval.json")
if os.path.exists(best_eval_path):
    with open(best_eval_path, "r") as f:
        best_eval = json.load(f)
    
    print("="*70)
    print("BEST MODEL PERFORMANCE (Selected based on Test Set Keyterm F1@10)")
    print("="*70)
    print(f"\nModel Information:")
    print(f"  Base Model:        {best_eval.get('model_name', 'N/A')}")
    print(f"  Best Epoch:        {best_eval.get('epoch', 'N/A')}")
    print(f"  Optimization Goal: {best_eval.get('optimization_goal', 'keyterm_prediction')}")
    
    metrics = best_eval.get('metrics', {})
    print(f"\nTest Set Performance - Top Keyterms Predicted:")
    print(f"  Keyterm Precision@5:  {metrics.get('keyterm_precision@5', 0.0):.6f}")
    print(f"  Keyterm Recall@5:     {metrics.get('keyterm_recall@5', 0.0):.6f}")
    print(f"  Keyterm F1@5:         {best_eval.get('keyterm_f1@5', 0.0):.6f}")
    print(f"  Keyterm Precision@10: {metrics.get('keyterm_precision@10', 0.0):.6f}")
    print(f"  Keyterm Recall@10:    {metrics.get('keyterm_recall@10', 0.0):.6f}")
    print(f"  Keyterm F1@10:        {best_eval.get('keyterm_f1@10', 0.0):.6f} ⭐ (selection metric)")
    print(f"\n  Keyword Precision@10: {metrics.get('keyword_precision@10', 0.0):.6f}")
    print(f"  Keyword Recall@10:    {metrics.get('keyword_recall@10', 0.0):.6f}")
    print(f"  Keyword F1@10:        {metrics.get('keyword_f1@10', 0.0):.6f}")
    
    dataset_info = best_eval.get('dataset_info', {})
    print(f"\nDataset Configuration:")
    print(f"  Target Role:      {dataset_info.get('target_role', 'N/A')}")
    print(f"  History Turns:    {dataset_info.get('history_turns', 'N/A')}")
    print(f"  Max Keywords:     {dataset_info.get('max_keywords', 'N/A')}")
    print(f"  Max Keyterms:     {dataset_info.get('max_keyterms', 'N/A')}")
    print(f"  Test Examples:   {dataset_info.get('num_test_examples', 'N/A')}")
    
    print(f"\nModel Saved To:")
    print(f"  {os.path.join(OUTPUT_DIR, 'encoder_best')}")
    print(f"\nFull Details:")
    print(f"  {best_eval_path}")
    print("="*70)
else:
    print(f"Best eval file not found at: {best_eval_path}")
    print("Training may still be in progress or failed.")

In [ ]:
# Demo: Generate keyterms for a sample conversation history
# This uses the best model (selected based on test set keyterm F1@10) to predict keyterms
RUN_DIR = OUTPUT_DIR
!python -m src.demo_offline --run "$RUN_DIR" --encoder_subdir encoder_best --topk 10